In [71]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [72]:
import pandas as pd
import pprint

In [73]:
from eml_transformer.ingestion.sources.iem_afos import IEMAFOSSource

source = IEMAFOSSource(
    product_types=["SPS", ],
    wfos=["IND"],
    limit=3, 
)

raw = source.fetch_raw(
    from_date="2024-01-01",
    to_date="2025-01-05",
)

In [74]:
print("RAW RESPONSES:", len(raw))
print("RAW KEYS:", raw[0].keys())

RAW RESPONSES: 1
RAW KEYS: dict_keys(['pil', 'response'])


In [75]:
records = source.parse_records(raw)

In [76]:
print("PARSED RECORDS:", len(records))

for i, record in enumerate(records):
    print("\n" + "=" * 80)
    print(f"RECORD {i + 1}")
    print("=" * 80)

    raw_text = record["raw_text"]
    header = source._parse_header(raw_text)
    sections = source._parse_sections(raw_text)
    issued = source._extract_issued_text(raw_text)

    print("PIL:", record["pil"])
    print("HEADER:", header)
    print("ISSUED:", issued)
    print("SECTIONS:", list(sections.keys()))
    print("RAW LEN:", len(raw_text))

    print("\nFIRST 10 LINES:")
    print("\n".join(raw_text.splitlines()[:10]))

    print("\nKEY MESSAGES:")
    print(sections.get("KEY MESSAGES", "MISSING")[:1000])

PARSED RECORDS: 3

RECORD 1
PIL: SPSIND
HEADER: {'raw_id': '236', 'wmo': 'WWUS83', 'wmo_header': 'WWUS83 KIND 021957', 'office': 'KIND', 'issued_code': '021957', 'pil': 'SPSIND'}
ISSUED: 257 PM EST Thu Jan 2 2025
SECTIONS: []
RAW LEN: 1704

FIRST 10 LINES:
236 
WWUS83 KIND 021957
SPSIND

Special Weather Statement
National Weather Service Indianapolis IN
257 PM EST Thu Jan 2 2025

INZ021-028>031-035>049-054>057-063>065-072-031000-
Carroll-Warren-Tippecanoe-Clinton-Howard-Fountain-Montgomery-

KEY MESSAGES:
MISSING

RECORD 2
PIL: SPSIND
HEADER: {'raw_id': '231', 'wmo': 'WWUS83', 'wmo_header': 'WWUS83 KIND 020853', 'office': 'KIND', 'issued_code': '020853', 'pil': 'SPSIND'}
ISSUED: 353 AM EST Thu Jan 2 2025
SECTIONS: []
RAW LEN: 2347

FIRST 10 LINES:
231 
WWUS83 KIND 020853
SPSIND

Special Weather Statement
National Weather Service Indianapolis IN
353 AM EST Thu Jan 2 2025

INZ021-028>031-035>049-051>057-060>065-067>072-022130-
Carroll-Warren-Tippecanoe-Clinton-Howard-Fountain-Montgomery-

In [77]:
standardized = [
    source.standardize_record(record)
    for record in records
]

for i, record in enumerate(standardized):
    print("\n" + "=" * 80)
    print(f"STANDARDIZED {i + 1}")
    print("=" * 80)

    print("record_id:", record.record_id)
    print("title:", record.title)
    print("published_at:", record.published_at)
    print("region:", record.region)
    print("categories:", record.categories)
    print("text len:", len(record.text or ""))
    print("raw sections:", list(record.metadata.get("sections", {}).keys()))

    print("\nTEXT PREVIEW:")
    print((record.text or "")[:1000])


STANDARDIZED 1
record_id: 733eeca6e0b55f9ee4d78651c29c84f2fe9f4fcb62ba6bb70bae7dfcd2d047a9
title: SPS | KIND | 257 PM EST Thu Jan 2 2025
published_at: 2025-01-02T19:57:00+00:00
region: IND
categories: ['weather', 'nws', 'iem', 'afos', 'sps']
text len: 1704
raw sections: []

TEXT PREVIEW:
236 
WWUS83 KIND 021957
SPSIND

Special Weather Statement
National Weather Service Indianapolis IN
257 PM EST Thu Jan 2 2025

INZ021-028>031-035>049-054>057-063>065-072-031000-
Carroll-Warren-Tippecanoe-Clinton-Howard-Fountain-Montgomery-
Boone-Tipton-Hamilton-Madison-Delaware-Randolph-Vermillion-Parke-
Putnam-Hendricks-Marion-Hancock-Henry-Morgan-Johnson-Shelby-Rush-
Brown-Bartholomew-Decatur-Jennings-
Including the cities of Delphi, Flora, Williamsport, 
West Lebanon, Lafayette, West Lafayette, Frankfort, Kokomo, 
Attica, Covington, Veedersburg, Crawfordsville, Lebanon, 
Zionsville, Tipton, Fishers, Carmel, Noblesville, Anderson, 
Muncie, Winchester, Union City, Farmland, Parker City, Clinton, 
Fair

### Checking Stored

In [78]:
import json

path = "../data/bronze/source=iem_afos/records.jsonl"

with open(path, "r", encoding="utf-8") as f:
    records = [
        json.loads(line)
        for line in f
        if line.strip()
    ]

print("records loaded:", len(records))

standardized = [
    source.standardize_record(record['raw'])
    for record in records
]

for i, record in enumerate(standardized):
    # print("\n" + "=" * 80)
    # print(f"STANDARDIZED {i + 1}")
    # print("=" * 80)

    # print("record_id:", record.record_id)
    # print("title:", record.title)
    # print("published_at:", record.published_at)
    # print("region:", record.region)
    # print("categories:", record.categories)
    # print("text len:", len(record.text or ""))

    sections = record.metadata.get("sections", {}) if record.metadata else {}
    print("raw sections:", list(sections.keys()))

    # print("\nTEXT PREVIEW:")
    # print((record.text or "")[:1000])

records loaded: 4562
raw sections: ['KEY MESSAGES', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'FORECAST UPDATE', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'FORECAST UPDATE', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'FORECAST UPDATE', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
raw sections: ['KEY MESSAGES', 'DISCUSSION', 'AVIATION', 'IND WATCHES/WARNINGS/ADVISORIES']
ra

In [79]:
df = pd.read_parquet('../data/silver/source=iem_afos/records.parquet')

In [80]:
df.columns

Index(['record_id', 'source', 'source_type', 'title', 'text', 'published_at',
       'retrieved_at', 'url', 'region', 'categories', 'metadata', 'raw'],
      dtype='object')

In [81]:
row = df.iloc[0]

for col in df.columns:
    print("\n" + "=" * 80)
    print(col.upper())
    print("=" * 80)
    print(row[col])


RECORD_ID
8d33de2da993cf47e11e2ef1b94ca3d5913bb8b3a539c224903fc5f5727ee9f4

SOURCE
iem_afos

SOURCE_TYPE
api

TITLE
WSW | KLSX | 134 AM CST Sun Mar 1 2026

TEXT
589 WWUS43 KLSX 010734 WSWLSX URGENT - WINTER WEATHER MESSAGE National Weather Service St Louis MO 134 AM CST Sun Mar 1 2026 ILZ058-059-095-097>100-MOZ018-019-026-027-034>036-042-051-052-060- 061-063-064-012245- /O.NEW.KLSX.WW.Y.0002.260302T0000Z-260302T1200Z/ Greene IL-Macoupin IL-Adams IL-Pike IL-Calhoun IL-Jersey IL- Madison IL-Knox MO-Lewis MO-Shelby MO-Marion MO-Monroe MO-Ralls MO-Pike MO-Audrain MO-Montgomery MO-Lincoln MO-Warren MO-Saint Charles MO-Saint Louis MO-Saint Louis City MO- Including the cities of Hannibal, Saint Charles, Saint Louis, Pittsfield, Alton, Quincy, Bowling Green, Mexico, and Edwardsville 134 AM CST Sun Mar 1 2026 ...WINTER WEATHER ADVISORY IN EFFECT FROM 6 PM THIS EVENING TO 6 AM CST MONDAY... * WHAT...Wet snow expected. Total snow accumulations up to one inch with a localized area of 2 to 4 inche

In [82]:
import json

In [83]:
for i, row in df.head(5).iterrows():
    print("\n" + "#" * 100)

    print("TITLE:")
    print(row["title"])

    print("\nPUBLISHED_AT:")
    print(row["published_at"])

    print("\nREGION:")
    print(row["region"])

    print("\nTEXT PREVIEW:")
    print(row["text"][:1000])

    print("\nTEXT LENGTH:")
    print(len(row["text"]))


    print("\nSECTION KEYS:")
    print(row["metadata"].get("sections", {}).keys())


####################################################################################################
TITLE:
WSW | KLSX | 134 AM CST Sun Mar 1 2026

PUBLISHED_AT:
2026-03-01T07:34:00+00:00

REGION:
LSX

TEXT PREVIEW:
589 WWUS43 KLSX 010734 WSWLSX URGENT - WINTER WEATHER MESSAGE National Weather Service St Louis MO 134 AM CST Sun Mar 1 2026 ILZ058-059-095-097>100-MOZ018-019-026-027-034>036-042-051-052-060- 061-063-064-012245- /O.NEW.KLSX.WW.Y.0002.260302T0000Z-260302T1200Z/ Greene IL-Macoupin IL-Adams IL-Pike IL-Calhoun IL-Jersey IL- Madison IL-Knox MO-Lewis MO-Shelby MO-Marion MO-Monroe MO-Ralls MO-Pike MO-Audrain MO-Montgomery MO-Lincoln MO-Warren MO-Saint Charles MO-Saint Louis MO-Saint Louis City MO- Including the cities of Hannibal, Saint Charles, Saint Louis, Pittsfield, Alton, Quincy, Bowling Green, Mexico, and Edwardsville 134 AM CST Sun Mar 1 2026 ...WINTER WEATHER ADVISORY IN EFFECT FROM 6 PM THIS EVENING TO 6 AM CST MONDAY... * WHAT...Wet snow expected. Total snow accumulatio

In [84]:
print(df.isna().sum())

record_id       0
source          0
source_type     0
title           0
text            0
published_at    0
retrieved_at    0
url             0
region          0
categories      0
metadata        0
raw             0
dtype: int64


In [85]:
df["published_at"].head(20)

0     2026-03-01T07:34:00+00:00
1     2026-03-01T08:16:00+00:00
2     2026-03-01T08:32:00+00:00
3     2026-03-01T09:20:00+00:00
4     2026-03-01T09:32:00+00:00
5     2026-03-01T09:50:00+00:00
6     2026-03-01T09:51:00+00:00
7     2026-03-01T10:03:00+00:00
8     2026-03-01T10:27:00+00:00
9     2026-03-01T18:02:00+00:00
10    2026-03-01T19:03:00+00:00
11    2026-03-01T19:12:00+00:00
12    2026-03-01T19:39:00+00:00
13    2026-03-01T19:43:00+00:00
14    2026-03-01T19:46:00+00:00
15    2026-03-01T20:09:00+00:00
16    2026-03-01T20:23:00+00:00
17    2026-03-01T20:50:00+00:00
18    2026-03-01T20:53:00+00:00
19    2026-03-01T21:08:00+00:00
Name: published_at, dtype: object

In [86]:
df.applymap(type).head(1).T

/tmp/ipykernel_41829/4094605693.py:1: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  df.applymap(type).head(1).T


,0
record_id,<class 'str'>
source,<class 'str'>
source_type,<class 'str'>
title,<class 'str'>
text,<class 'str'>
published_at,<class 'str'>
retrieved_at,<class 'str'>
url,<class 'str'>
region,<class 'str'>
categories,<class 'numpy.ndarray'>


In [87]:
df["metadata"].apply(
    lambda r: list(r.get("sections", {}).keys())
).value_counts()

metadata
[DAY ONE, DAYS TWO THROUGH SEVEN, SPOTTER INFORMATION STATEMENT, KEY MESSAGES, SHORT TERM /THROUGH SATURDAY/, LONG TERM /SATURDAY NIGHT THROUGH THURSDAY/, AVIATION /06Z TAFS THROUGH 06Z SATURDAY/, DMX WATCHES/WARNINGS/ADVISORIES, DISCUSSION, AVIATION /12Z TAFS THROUGH 12Z SATURDAY/, IWX WATCHES/WARNINGS/ADVISORIES, AVIATION /18Z TAFS THROUGH 18Z SATURDAY/, SHORT TERM, LONG TERM, AVIATION, LSX WATCHES/WARNINGS/ADVISORIES, UPDATE, AVIATION /00Z TAFS THROUGH 00Z SUNDAY/, AVIATION /06Z TAFS THROUGH 06Z SUNDAY/, LONG TERM /SATURDAY NIGHT THROUGH FRIDAY/, DVN WATCHES/WARNINGS/ADVISORIES, SHORT TERM /THROUGH MONDAY/, LONG TERM /MONDAY NIGHT THROUGH FRIDAY/, AVIATION /18Z TAFS THROUGH 18Z SUNDAY/, AVIATION /12Z TAFS THROUGH 12Z SUNDAY/, SHORT TERM /THROUGH TONIGHT/, LONG TERM /SUNDAY THROUGH FRIDAY/, AVIATION /00Z TAFS THROUGH 00Z MONDAY/, AVIATION /06Z TAFS THROUGH 06Z MONDAY/, ILX WATCHES/WARNINGS/ADVISORIES, SHORT TERM /THROUGH MONDAY NIGHT/, LONG TERM /TUESDAY THROUGH SATURDAY/, L